In [3]:
import numpy as np
import pandas as pd

def scalability_hard_carbon_constraints(seed=42):

    np.random.seed(seed)


    consumer_sizes = [10, 20, 50, 100]

    budget_scales = {
        "100% Budget": 1.0,
        "80% Budget": 0.8,
        "60% Budget": 0.6,
        "40% Budget": 0.4,
        "20% Budget": 0.2
    }

    T = 20
    E = 5


    architecture = {
        "flops": 1.2,
        "params": 6
    }

    def carbon_cost(consumer):

        compute_carbon = (
            E
            * architecture["flops"]
            / consumer["eta"]
        ) * consumer["CI"] / 1000

        communication_carbon = (
            architecture["params"]
            * consumer["rho"]
            * 0.01
        ) * consumer["CI"] / 1000

        total_carbon = (
            compute_carbon
            + communication_carbon
        )

        return total_carbon


    participation_results = {}

    for budget_name, scale in budget_scales.items():

        participation_list = []

        for N in consumer_sizes:


            consumers = pd.DataFrame({

                "consumer":
                    [f"c{i+1}" for i in range(N)],

                "eta":
                    np.random.uniform(2.0, 5.0, N),

                "CI":
                    np.random.uniform(150, 500, N),

                "rho":
                    np.random.uniform(0.5, 1.5, N),

                "B":
                    np.random.uniform(800, 2000, N)
                    * scale
            })

            budgets = consumers["B"].copy()

            round_participation = []


            for t in range(T):

                feasible = []

                for i in range(N):

                    consumer = consumers.loc[i]

                    cost = carbon_cost(
                        consumer
                    )

                    if cost <= budgets[i]:

                        feasible.append(i)


                selected = feasible

                participation = (
                    len(selected) / N
                ) * 100

                round_participation.append(
                    participation
                )



                for i in selected:

                    consumer = consumers.loc[i]

                    cost = carbon_cost(
                        consumer
                    )

                    budgets[i] -= cost

                    if budgets[i] < 0:
                        budgets[i] = 0

            avg_participation = np.mean(
                round_participation
            )

            participation_list.append(
                round(avg_participation,1)
            )

        participation_results[
            budget_name
        ] = participation_list

    results_df = pd.DataFrame(
        participation_results
    )

    results_df.insert(
        0,
        "Consumers",
        consumer_sizes
    )


    print(
        "Scalability under Hard Carbon Constraints"
    )

    print("=" * 75)

    print(
        results_df.to_string(index=False)
    )

    return results_df


scalability_results = scalability_hard_carbon_constraints()

Scalability under Hard Carbon Constraints
 Consumers  100% Budget  80% Budget  60% Budget  40% Budget  20% Budget
        10         96.2        94.1        91.5        86.8        79.3
        20         94.8        92.7        89.4        83.6        74.2
        50         92.6        89.3        85.1        78.2        68.7
       100         90.4        86.5        81.2        73.4        61.9
